# Drug type counts

This notebook aggregates the number of charges involving each drug type, both including and excluding excluded cases.

In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd().resolve()
if not (repo_root / 'featureExtraction').exists():
    repo_root = repo_root.parent

for env_path in (repo_root / 'featureExtraction' / '.env', repo_root / 'featureVerification' / '.env.local', repo_root / '.env'):
    if env_path.exists():
        load_dotenv(env_path)

from evaluate_verified_sentences import get_collection

verified_collection, _ = get_collection()
query = {'is_verified': True}
projection = {'exclude': 1, 'trials': 1}
docs = list(verified_collection.find(query, projection))

counts_all = defaultdict(int)
counts_excluding_excluded = defaultdict(int)

for doc in docs:
    trials = (doc.get('trials') or {}).get('trials') or []
    is_excluded = bool(doc.get('exclude'))
    for trial in trials:
        seen_drugs = set()
        for drug in trial.get('drugs') or []:
            drug_type = drug.get('drug_type')
            if not drug_type:
                continue

            if drug_type == 'Other':
                drug_label = (drug.get('other_drug_type') or 'Unknown').strip()
                if not drug_label:
                    drug_label = 'Unknown'
                group_key = f'Other: {drug_label}'
            else:
                group_key = drug_type

            if group_key in seen_drugs:
                continue
            seen_drugs.add(group_key)
            counts_all[group_key] += 1
            if not is_excluded:
                counts_excluding_excluded[group_key] += 1

drug_types = sorted(set(counts_all) | set(counts_excluding_excluded))
rows = []
for drug_type in drug_types:
    rows.append({
        'drug_type': drug_type,
        'count_all_charges': counts_all.get(drug_type, 0),
        'count_excluding_excluded_charges': counts_excluding_excluded.get(drug_type, 0),
    })

counts_df = pd.DataFrame(rows).sort_values(['count_all_charges', 'drug_type'], ascending=[False, True]).reset_index(drop=True)
output_dir = repo_root / 'notebooks'
output_dir.mkdir(exist_ok=True)
try:
    counts_df.to_excel(output_dir / 'drug_type_counts.xlsx', index=False)
except Exception as exc:
    print(f'Excel export skipped: {exc}')

print(f'Aggregated {len(counts_df)} drug types')
counts_df.head(20)

/Users/cxiang/Projects/drug-trafficing-sentence-predictor/featureExtraction/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Aggregated 38 drug types


,drug_type,count_all_charges,count_excluding_excluded_charges
0,Cocaine,1826,1655
1,Ketamine,968,884
2,Methamphetamine,795,704
3,Heroin,385,331
4,Cannabis,209,191
5,Ecstasy,90,80
6,Other: Midazolam,84,81
7,Fluorodeschloroketamine,54,48
8,THC/CBD,29,28
9,GHB/GBL,11,10
